In [1]:
import re
import textwrap
from datasketch import MinHash

text1 = (
    "Иней — мелкие кристаллы льда, выделившегося из влажного газа на охлаждённых предметах; "
    "вид твёрдых атмосферных осадков. Представляет собой тонкий слой кристаллического водного "
    "льда различной мощности, нарастающего на поверхности земли и наземных предметах "
    "при отрицательной температуре почвы, малооблачном небе и слабом ветре."
)
text2 = (
    "Изморозь — вид атмосферных осадков, представляет собой кристаллические или зернистые "
    "отложения льда на тонких и длинных предметах при влажной морозной погоде. На поверхности "
    "предметов, крышах зданий и автомобилей изморозь отлагается очень слабо."
)

In [2]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def get_char_ngrams(text, n):
    text = " " * (n - 1) + text + " " * (n - 1)
    return {text[i:i + n] for i in range(len(text) - n + 1)}

def get_jaccard(set1, set2):
    if len(set1) == 0 and len(set2) == 0:
        return 1.0
    return len(set1 & set2) / len(set1 | set2)

In [3]:
norm1 = normalize_text(text1)
norm2 = normalize_text(text2)

print(textwrap.fill(norm1, width=100), textwrap.fill(norm2, width=100), sep="\n\n")

иней мелкие кристаллы льда выделившегося из влажного газа на охлаждённых предметах вид твёрдых
атмосферных осадков представляет собой тонкий слой кристаллического водного льда различной мощности
нарастающего на поверхности земли и наземных предметах при отрицательной температуре почвы
малооблачном небе и слабом ветре

изморозь вид атмосферных осадков представляет собой кристаллические или зернистые отложения льда на
тонких и длинных предметах при влажной морозной погоде на поверхности предметов крышах зданий и
автомобилей изморозь отлагается очень слабо


In [4]:
n = 3
ngrams1 = get_char_ngrams(norm1, n)
ngrams2 = get_char_ngrams(norm2, n)
common = ngrams1 & ngrams2

print(
    f"Количество {n}-грамм в тексте 1: {len(ngrams1)}",
    f"Количество {n}-грамм в тексте 2: {len(ngrams2)}\n",
    f"Примеры {n}-грамм текста 1: {list(sorted(ngrams1))[:10]}",
    f"Примеры {n}-грамм текста 2: {list(sorted(ngrams2))[:10]}\n",
    f"Примеры общих {n}-грамм: {list(sorted(common))[:10]}",
    f"Количество общих {n}-грамм: {len(common)}",
    sep="\n"
)

Количество 3-грамм в тексте 1: 256
Количество 3-грамм в тексте 2: 201

Примеры 3-грамм текста 1: ['  и', ' ат', ' ве', ' ви', ' вл', ' во', ' вы', ' га', ' зе', ' и ']
Примеры 3-грамм текста 2: ['  и', ' ав', ' ат', ' ви', ' вл', ' дл', ' зд', ' зе', ' и ', ' из']

Примеры общих 3-грамм: ['  и', ' ат', ' ви', ' вл', ' зе', ' и ', ' из', ' кр', ' ль', ' мо']
Количество общих 3-грамм: 112


In [5]:
num_perm = 128
min_hash1 = MinHash(num_perm=num_perm)
min_hash2 = MinHash(num_perm=num_perm)

for g in ngrams1:
    min_hash1.update(g.encode("utf-8"))
for g in ngrams2:
    min_hash2.update(g.encode("utf-8"))

print(
    "Примеры значений MinHash-представления текста 1:",
    [min_hash1.digest()[i] for i in range(5)],
    "",
    "Примеры значений MinHash-представления текста 2:",
    [min_hash2.digest()[i] for i in range(5)],
    sep="\n"
)

Примеры значений MinHash-представления текста 1:
[np.uint64(18460455), np.uint64(7878845), np.uint64(7113949), np.uint64(21261234), np.uint64(10731529)]

Примеры значений MinHash-представления текста 2:
[np.uint64(20931045), np.uint64(3376189), np.uint64(16032978), np.uint64(21261234), np.uint64(10731529)]


In [6]:
jaccard_exact = get_jaccard(ngrams1, ngrams2)
jaccard_approx = min_hash1.jaccard(min_hash2)

print(f"Точный коэффициент Жаккара: {jaccard_exact:.4f}")
print(f"Приближённый коэффициент Жаккара: {jaccard_approx:.4f}")

Точный коэффициент Жаккара: 0.3246
Приближённый коэффициент Жаккара: 0.3516


In [ ]:
print(f"Степень сходства текстов: {jaccard_approx * 100:.1f}%")

if jaccard_approx > 0.8:
    result = "очень похожи"
elif jaccard_approx > 0.5:
    result = "достаточно похожи"
elif jaccard_approx > 0.2:
    result = "немного похожи"
else:
    result = "почти не похожи"

print(f"Вывод: тексты {result}")

Степень сходства текстов: 35.2%
Вывод: тексты немного похожи
